In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
file_path = '/kaggle/input/nids-for-iot/inids.csv'
dataset = pd.read_csv(file_path)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/nids-for-iot/inids.csv'

In [ ]:
print(dataset.shape)
dataset.head()

In [ ]:
dataset.describe()

In [ ]:
dataset.info()

In [ ]:
nan_col = dataset.columns[dataset.isna().any()].tolist()
print('Columns with null values:\n',nan_col,'\n')

infinity = [np.inf, -np.inf]
inf_columns = []
for col in (dataset.columns):
    if dataset[col].isin(infinity).any():
        inf_columns.append(col)
print('\n Columns with inf or -inf values:\n',inf_columns,'\n')

In [ ]:
dataset.replace([np.inf, -np.inf], np.nan,inplace=True)

In [ ]:
print('Filling \'Flow Pkts/s\' nan values with mean :',dataset['Flow_Pkts/s'].mean())
print('Filling \'Flow Byts/s\' nan values with mean :',dataset['Flow_Byts/s'].mean())

dataset['Flow_Byts/s'].fillna(dataset['Flow_Byts/s'].mean(),inplace=True)
dataset['Flow_Pkts/s'].fillna(dataset['Flow_Pkts/s'].mean(),inplace=True)


In [ ]:
drop_lst = []
catfeat = []
for col in (dataset.columns):
    if len(dataset[col].unique())==1:
        drop_lst.append(col)
    elif len(dataset[col].unique())<=2:
        catfeat.append(col)

catfeat.append('y')     
print('Columns with single value:\n',np.array(drop_lst),'\n')
print('Categorical columns:\n',np.array(catfeat),'\n')

In [ ]:
labels = dataset.value_counts('Label')
print(labels)
labels.plot(kind='bar')

In [ ]:
dataset.value_counts('Flow_Byts/s')

In [ ]:
dataset.value_counts('Flow_Pkts/s')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# performing the data exploratory analysis.
selected_attributes = [
    "Flow_Duration",
    "Tot_Fwd_Pkts",
    "Tot_Bwd_Pkts",
    "Flow_Byts/s",
    "Flow_Pkts/s",
]
# Remove non-finite or infinite values from Flow_Pkts/s
dataset = dataset[np.isfinite(dataset["Flow_Pkts/s"])]

# Create a histogram of Flow_Pkts/s with a limited range
plt.hist(dataset["Flow_Pkts/s"], bins=10, range=(0, 200))  # Adjust the range as needed
plt.xlabel("Flow Packets/s")
plt.ylabel("Frequency")
plt.title("Histogram: Flow Packets/s")
plt.show()

# Create a bar plot for Tot_Bwd_Pkts based on Label
label_counts = dataset.groupby("Label")["Tot_Bwd_Pkts"].mean()
label_counts.plot(kind="bar")
plt.xlabel("Label")
plt.ylabel("Mean Total Backward Packets")
plt.title("Bar Plot: Mean Total Backward Packets by Label")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Group data by 'Cate' and 'Sub_Cat' and calculate the count
grouped = dataset.groupby(['Cate', 'Sub_Cat']).size().unstack()

# Plot a stacked bar chart
ax = grouped.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title("Distribution of Sub_Cat within each Cate")
plt.xlabel("Cate")
plt.ylabel("Count")
plt.legend(title="Sub_Cat")
plt.xticks(rotation=45, ha="right")

# Display the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Group data by 'Label', 'Sub', and 'Sub_Cat' and calculate the count
grouped = dataset.groupby(['Label', 'Cate', 'Sub_Cat']).size().unstack()

# Plot a grouped bar chart
ax = grouped.plot(kind='bar', figsize=(12, 6))
plt.title("Distribution of Sub_Cat within each Label and Sub")
plt.xlabel("Label and Sub")
plt.ylabel("Count")
plt.legend(title="Sub_Cat")
plt.xticks(rotation=45, ha="right")

# Display the plot
plt.tight_layout()
plt.show()


In [ ]:
# performing the encoding to the label encoder to convert the object into numeric value i,e normal and anomaly.
from sklearn.preprocessing import LabelEncoder
import pandas as pd
label_encoder = LabelEncoder()
dataset["Label_Encoded"] = label_encoder.fit_transform(dataset["Label"])
dataset.drop(columns=['Label'], inplace=True)


In [ ]:
print(dataset["Label_Encoded"])

In [ ]:
print(dataset.Cate)

In [ ]:
# similarly for category and sub category
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
dataset['Cate'] = label_encoder.fit_transform(dataset['Cate'])
dataset['Sub_Cat'] = label_encoder.fit_transform(dataset['Sub_Cat'])


In [ ]:
dataset

In [ ]:
dataset.Sub_Cat

In [ ]:
import pandas as pd
flow_id_encoder = LabelEncoder()
src_ip_encoder = LabelEncoder()
dst_ip_encoder = LabelEncoder()

# Fit and transform 'Flow_ID' and 'Src_IP' columns
encoded_flow_id = flow_id_encoder.fit_transform(dataset['Flow_ID'])
encoded_src_ip = src_ip_encoder.fit_transform(dataset['Src_IP'])
encoded_dst_ip = dst_ip_encoder.fit_transform(dataset['Dst_IP'])

# Update the dataset with encoded values
dataset['Flow_ID'] = encoded_flow_id
dataset['Src_IP'] = encoded_src_ip
dataset['Dst_IP'] = encoded_dst_ip
dataset.to_csv('dataset.csv', index=False)

In [ ]:
# to remove the column havaing all value zero
columns_to_remove = []
for column in dataset.columns:
    if dataset[column].all() == 0:
        columns_to_remove.append(column)


In [ ]:
dataset.info()

In [ ]:
# performing encoding for the timestamp.
import pandas as pd
from sklearn.preprocessing import LabelEncoder

timestamp_column_name = 'Timestamp'
dataset[timestamp_column_name] = pd.to_datetime(dataset[timestamp_column_name], format='%d-%m-%Y %H:%M')
encoder = LabelEncoder()
dataset['Encoded_Timestamp'] = encoder.fit_transform(dataset[timestamp_column_name])
dataset.drop(columns=[timestamp_column_name], inplace=True)
print(dataset)


In [ ]:
import pandas as pd
import numpy as np

problematic_columns = []
for column in dataset.columns:
    if pd.api.types.is_numeric_dtype(dataset[column]): 
        if np.any(np.isinf(dataset[column])) or np.any(dataset[column] > 1e6):  
            problematic_columns.append(column)

print("Problematic Columns:", problematic_columns)


In [ ]:
# List of columns to remove so that we can easily perform the normalization
columns_to_remove = ['Flow_Byts/s', 'Flow_Pkts/s', 'Fwd_Pkts/s']
dataset.drop(columns=columns_to_remove, inplace=True)
print(dataset)

In [ ]:
from sklearn.preprocessing import StandardScaler
# Get a list of all numerical attribute names except 'Label_Encoded'
numerical_attributes = [col for col in dataset.columns if col != 'Label_Encoded' and dataset[col].dtype != 'object']
# Create a subset of the data with only the numerical attributes
X_subset = dataset[numerical_attributes]
scaler = StandardScaler()
# Fit and transform the selected numerical attributes
X_normalized = scaler.fit_transform(X_subset)
dataset[numerical_attributes] = X_normalized


In [ ]:
dataset

In [ ]:
dataset.columns

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

features_to_select = [
    'Src_Port', 'Dst_Port', 'Protocol',
    'Flow_Duration', 'Tot_Fwd_Pkts', 'Tot_Bwd_Pkts', 'TotLen_Fwd_Pkts',
    'TotLen_Bwd_Pkts', 'Fwd_Pkt_Len_Max', 'Fwd_Pkt_Len_Min',
    'Fwd_Pkt_Len_Mean', 'Fwd_Pkt_Len_Std', 'Bwd_Pkt_Len_Max',
    'Bwd_Pkt_Len_Min', 'Bwd_Pkt_Len_Mean', 'Bwd_Pkt_Len_Std',
    'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max', 'Flow_IAT_Min',
    'Fwd_IAT_Tot', 'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_IAT_Max',
    'Fwd_IAT_Min', 'Bwd_IAT_Tot', 'Bwd_IAT_Mean', 'Bwd_IAT_Std',
    'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_PSH_Flags', 'Bwd_PSH_Flags',
    'Fwd_URG_Flags', 'Bwd_URG_Flags', 'Fwd_Header_Len', 'Bwd_Header_Len',
    'Bwd_Pkts/s', 'Pkt_Len_Min', 'Pkt_Len_Max', 'Pkt_Len_Mean',
    'Pkt_Len_Std', 'Pkt_Len_Var', 'FIN_Flag_Cnt', 'SYN_Flag_Cnt',
    'RST_Flag_Cnt', 'PSH_Flag_Cnt', 'ACK_Flag_Cnt', 'URG_Flag_Cnt',
    'CWE_Flag_Count', 'ECE_Flag_Cnt', 'Down/Up_Ratio', 'Pkt_Size_Avg',
    'Fwd_Seg_Size_Avg', 'Bwd_Seg_Size_Avg', 'Fwd_Byts/b_Avg',
    'Fwd_Pkts/b_Avg', 'Fwd_Blk_Rate_Avg', 'Bwd_Byts/b_Avg',
    'Bwd_Pkts/b_Avg', 'Bwd_Blk_Rate_Avg', 'Subflow_Fwd_Pkts',
    'Subflow_Fwd_Byts', 'Subflow_Bwd_Pkts', 'Subflow_Bwd_Byts',
    'Init_Fwd_Win_Byts', 'Init_Bwd_Win_Byts', 'Fwd_Act_Data_Pkts',
    'Fwd_Seg_Size_Min', 'Active_Mean', 'Active_Std', 'Active_Max',
    'Active_Min', 'Idle_Mean', 'Idle_Std', 'Idle_Max', 'Idle_Min'
]

# Create a subset of the data with the selected features
selected_data = dataset[features_to_select + ['Label_Encoded']]

# Define features and target
X = selected_data.drop(['Label_Encoded'], axis=1)
y = selected_data['Label_Encoded']

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a random forest model to get feature importances
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Get feature importances
feature_importances = model.feature_importances_

# Select the top 40 features based on importance scores
k = 40
top_features_indices = feature_importances.argsort()[-k:][::-1]
top_features = [X_train.columns[i] for i in top_features_indices]

# Create a new dataset with the top 40 features and the target variable
selected_data_top40 = selected_data[top_features + ['Label_Encoded']]



In [ ]:
top_features

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

# Normalize the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

num_classes = len(y_train.unique())


# Build CNN component with dropout layer
cnn_input = Input(shape=(X_train.shape[1],))
cnn_layer = Dense(64, activation='relu')(cnn_input)
cnn_layer = Dropout(0.5)(cnn_layer)  

# Build Transformer component with dropout layer
transformer_input = Input(shape=(X_train_scaled.shape[1],))
transformer_layer = Dense(64, activation='relu')(transformer_input)
transformer_layer = Dropout(0.5)(transformer_layer)  

  

# Combine CNN and Transformer components
combined = Concatenate()([cnn_layer, transformer_layer])
output = Dense(num_classes, activation='softmax')(combined)

# Build and compile the hybrid model
hybrid_model = Model(inputs=[cnn_input, transformer_input], outputs=output)
hybrid_model.compile(optimizer=Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

from tensorflow.keras.callbacks import EarlyStopping

# Add early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Train the hybrid model with early stopping
history = hybrid_model.fit(
    [X_train, X_train_scaled], y_train,
    validation_data=([X_test_scaled, X_test_scaled], y_test),
    batch_size=10,
    epochs=5,  
    callbacks=[early_stopping]  
)


# Evaluate the model
test_loss, test_accuracy = hybrid_model.evaluate([X_test, X_test_scaled], y_test)

print("Test Accuracy:", test_accuracy)


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate, Conv1D, MaxPooling1D, Flatten, MultiHeadAttention, LayerNormalization, Add
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

# 1. Preprocessing & Reshaping
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape for Conv1D and Attention: (samples, features, 1)
X_train_reshaped = np.expand_dims(X_train_scaled, axis=-1)
X_test_reshaped = np.expand_dims(X_test_scaled, axis=-1)

num_features = X_train_reshaped.shape[1]
num_classes = len(np.unique(y_train))

# --- Model Components ---

input_layer = Input(shape=(num_features, 1))

# 2. CNN Branch (Spatial Feature Extraction)
cnn = Conv1D(filters=32, kernel_size=3, activation='relu', padding='same')(input_layer)
cnn = MaxPooling1D(pool_size=2)(cnn)
cnn = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(cnn)
cnn = Flatten()(cnn)
cnn = Dense(64, activation='relu')(cnn)

# 3. Transformer Branch (Self-Attention)
# Transformer expects (batch, seq_len, embed_dim). We'll treat features as the sequence.
# Multi-head attention
attention_out = MultiHeadAttention(num_heads=4, key_dim=16)(input_layer, input_layer)
attention_out = Dropout(0.1)(attention_out)
# Add & Norm (Residual Connection)
x = Add()([input_layer, attention_out])
x = LayerNormalization(epsilon=1e-6)(x)
# Feed Forward Part
ff_net = Dense(32, activation='relu')(x)
ff_net = Dense(1)(ff_net) # Bring back to original dim for residual
x = Add()([x, ff_net])
transformer_out = Flatten()(LayerNormalization(epsilon=1e-6)(x))
transformer_out = Dense(64, activation='relu')(transformer_out)

# 4. Hybrid Fusion
combined = Concatenate()([cnn, transformer_out])
combined = Dense(128, activation='relu')(combined)
combined = Dropout(0.3)(combined)

output = Dense(num_classes, activation='softmax')(combined)

# 5. Build and Compile
hybrid_model = Model(inputs=input_layer, outputs=output)
hybrid_model.compile(optimizer=Adam(learning_rate=1e-4), 
                     loss='sparse_categorical_crossentropy', 
                     metrics=['accuracy'])

# Train
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = hybrid_model.fit(
    X_train_reshaped, y_train,
    validation_data=(X_test_reshaped, y_test),
    batch_size=64,
    epochs=20,
    callbacks=[early_stopping]
)

# Evaluate
test_loss, test_accuracy = hybrid_model.evaluate(X_test_reshaped, y_test)
print(f"Final Test Accuracy: {test_accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot training history (accuracy and loss)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Make predictions on the test set
y_pred = hybrid_model.predict([X_test, X_test_scaled]).argmax(axis=1)

# Create a confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

# Print classification report
class_names = [str(label) for label in range(num_classes)]
print(classification_report(y_test, y_pred, target_names=class_names))


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
y_pred = hybrid_model.predict([X_test, X_test_scaled])
y_pred_classes = np.argmax(y_pred, axis=1)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred_classes)
print("Accuracy:", accuracy)

# Compute precision
precision = precision_score(y_test, y_pred_classes)
print("Precision:", precision)

# Compute recall
recall = recall_score(y_test, y_pred_classes)
print("Recall:", recall)

# Compute F1-score
f1 = f1_score(y_test, y_pred_classes)
print("F1-score:", f1)
fpr, tpr, _ = roc_curve(y_test, y_pred[:, 1])  
roc_auc = auc(fpr, tpr)
print("ROC AUC:", roc_auc)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()
